In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
save_dir = "/content/drive/MyDrive/NLP_Project_Preprocessing"

In [ ]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 77.8 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pickle
from gensim.models import Word2Vec

In [ ]:
X_train_pad = np.load(f"{save_dir}/X_train_pad.npy")
X_test_pad = np.load(f"{save_dir}/X_test_pad.npy")

y_train_sentiment = np.load(f"{save_dir}/y_train_sentiment.npy")
y_test_sentiment = np.load(f"{save_dir}/y_test_sentiment.npy")

In [ ]:
embedding_matrix = np.load(
    f"{save_dir}/embedding_matrix.npy"
)

In [ ]:
w2v_model = Word2Vec.load(
    f"{save_dir}/word2vec.model"
)

In [ ]:
with open(f"{save_dir}/word_index.pkl", "rb") as f:
    word_index = pickle.load(f)

vocab_size = len(word_index) + 2

In [ ]:
import joblib

stance_encoder = joblib.load(
    f"{save_dir}/stance_encoder.pkl"
)

In [ ]:
print("X_train_pad:", X_train_pad.shape)
print("X_test_pad:", X_test_pad.shape)

print("y_train_sentiment:", y_train_sentiment.shape)
print("y_test_sentiment:", y_test_sentiment.shape)

print("Embedding Matrix:", embedding_matrix.shape)

print("Vocabulary Size:", len(word_index))

X_train_pad: (1166475, 30)
X_test_pad: (291619, 30)
y_train_sentiment: (1166475,)
y_test_sentiment: (291619,)
Embedding Matrix: (81958, 100)
Vocabulary Size: 81956


##model 1 computed class wts

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train_sentiment)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_sentiment
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(0.8395355214264277), np.int64(1): np.float64(0.5681477790620028), np.int64(2): np.float64(20.508729363363045)}


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

rnn_model = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=False
    ),

    SimpleRNN(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
rnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [ ]:
history = rnn_model.fit(
    X_train_pad,
    y_train_sentiment,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 87s 10ms/step - accuracy: 0.3283 - loss: 1.0914 - val_accuracy: 0.3916 - val_loss: 1.0970
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 44s 5ms/step - accuracy: 0.3275 - loss: 1.0913 - val_accuracy: 0.3844 - val_loss: 1.1006
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 45s 5ms/step - accuracy: 0.3809 - loss: 1.0414 - val_accuracy: 0.3438 - val_loss: 1.0716
Epoch 4/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 45s 6ms/step - accuracy: 0.3795 - loss: 1.0832 - val_accuracy: 0.3873 - val_loss: 1.0726
Epoch 5/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 45s 6ms/step - accuracy: 0.4052 - loss: 1.0871 - val_accuracy: 0.3861 - val_loss: 1.0844


In [ ]:
test_loss, test_acc = rnn_model.evaluate(
    X_test_pad,
    y_test_sentiment,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - accuracy: 0.3452 - loss: 1.0712
Test Accuracy: 0.34522441029548645


In [ ]:
import numpy as np

y_pred_probs = rnn_model.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.43      0.85      0.57    115785
     Neutral       0.00      0.00      0.00    171094
    Positive       0.04      0.60      0.08      4740

    accuracy                           0.35    291619
   macro avg       0.16      0.48      0.22    291619
weighted avg       0.17      0.35      0.23    291619



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[ 97843      0  17942]
 [126920      0  44174]
 [  1909      0   2831]]


#model 2 class wts + trainable embeddings

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train_sentiment)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_sentiment
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(0.8395355214264277), np.int64(1): np.float64(0.5681477790620028), np.int64(2): np.float64(20.508729363363045)}


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

rnn_model_2 = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    SimpleRNN(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
rnn_model_2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = rnn_model_2.fit(
    X_train_pad,
    y_train_sentiment,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 64s 7ms/step - accuracy: 0.5054 - loss: 0.8386 - val_accuracy: 0.7131 - val_loss: 0.6662
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - accuracy: 0.7089 - loss: 0.5533 - val_accuracy: 0.6961 - val_loss: 0.7439
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - accuracy: 0.7368 - loss: 0.5004 - val_accuracy: 0.7412 - val_loss: 0.6305
Epoch 4/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - accuracy: 0.7560 - loss: 0.4649 - val_accuracy: 0.7369 - val_loss: 0.6181
Epoch 5/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 55s 7ms/step - accuracy: 0.7652 - loss: 0.4524 - val_accuracy: 0.7495 - val_loss: 0.5811
Epoch 6/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 66s 8ms/step - accuracy: 0.7747 - loss: 0.4310 - val_accuracy: 0.7524 - val_loss: 0.5889
Epoch 7/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 55s 7ms/step - accuracy: 0.7832 - loss: 0.4210 - val_accuracy: 0.7341 - val_loss: 0.6267


In [ ]:
test_loss, test_acc = rnn_model_2.evaluate(
    X_test_pad,
    y_test_sentiment,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - accuracy: 0.7514 - loss: 0.5758
Test Accuracy: 0.7513879537582397


In [ ]:
import numpy as np

y_pred_probs = rnn_model_2.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.72      0.82      0.77    115785
     Neutral       0.86      0.70      0.77    171094
    Positive       0.21      0.84      0.33      4740

    accuracy                           0.75    291619
   macro avg       0.59      0.79      0.62    291619
weighted avg       0.79      0.75      0.76    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)
print(cm)

[[ 94904  19575   1306]
 [ 37025 120248  13821]
 [    54    719   3967]]


In [ ]:
rnn_model_2.save(f"{save_dir}/rnn_sentiment.keras")

In [ ]:
import os

print(os.listdir(save_dir))

['sentiment_encoder.pkl', 'stance_encoder.pkl', 'train_idx.npy', 'test_idx.npy', 'y_train_stance.npy', 'y_test_stance.npy', 'y_train_sentiment.npy', 'y_test_sentiment.npy', 'X_train.csv', 'X_test.csv', 'X_train_tokens.pkl', 'X_test_tokens.pkl', 'word_index.pkl', 'word2vec.model', 'X_train_pad.npy', 'X_test_pad.npy', 'embedding_matrix.npy', 'rnn_stance.keras', 'rnn_sentiment.keras']


##model 3 with wts= 1,0.7,10 and trainable=true

In [ ]:
class_weights = {
    0: 1.0,
    1: 0.7,
    2: 10.0
}

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

rnn_model_3 = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    SimpleRNN(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
rnn_model_3.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = rnn_model_3.fit(
    X_train_pad,
    y_train_sentiment,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 61s 7ms/step - accuracy: 0.5468 - loss: 0.9985 - val_accuracy: 0.5884 - val_loss: 0.8751
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 54s 7ms/step - accuracy: 0.5723 - loss: 0.9983 - val_accuracy: 0.5884 - val_loss: 0.8916


In [ ]:
test_loss, test_acc = rnn_model_3.evaluate(
    X_test_pad,
    y_test_sentiment,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - accuracy: 0.5867 - loss: 0.8748
Test Accuracy: 0.586703896522522


In [ ]:
import numpy as np

y_pred_probs = rnn_model_3.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.72      0.89      0.80    115785
     Neutral       0.89      0.77      0.82    171094
    Positive       0.00      0.00      0.00      4740

    accuracy                           0.81    291619
   macro avg       0.54      0.55      0.54    291619
weighted avg       0.81      0.81      0.80    291619



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[103501  12284      0]
 [ 39636 131458      0]
 [   333   4407      0]]


##model 4 (0.9,0.7,14) + trainable = true

In [ ]:
class_weights = {
    0: 0.9,
    1: 0.7,
    2: 14.0
}

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

rnn_model_4 = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    SimpleRNN(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
rnn_model_4.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = rnn_model_4.fit(
    X_train_pad,
    y_train_sentiment,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 70s 8ms/step - accuracy: 0.5620 - loss: 1.0519 - val_accuracy: 0.5883 - val_loss: 0.9850
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 63s 8ms/step - accuracy: 0.5574 - loss: 1.0480 - val_accuracy: 0.5300 - val_loss: 0.9282


In [ ]:
test_loss, test_acc = rnn_model_4.evaluate(
    X_test_pad,
    y_test_sentiment,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 31s 3ms/step - accuracy: 0.5864 - loss: 0.9853
Test Accuracy: 0.5864261388778687


In [ ]:
import numpy as np

y_pred_probs = rnn_model_4.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 24s 3ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00    115785
     Neutral       0.59      1.00      0.74    171094
    Positive       0.30      0.02      0.03      4740

    accuracy                           0.59    291619
   macro avg       0.30      0.34      0.26    291619
weighted avg       0.35      0.59      0.43    291619



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[     0 115768     17]
 [     0 170940    154]
 [     0   4667     73]]
